# Mistral Live Runs


## Setup


In [ ]:
# Libraries
import json
import sys
from pathlib import Path
import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Pipeline
%load_ext autoreload
%autoreload 2

import backends
import flags
import run
import settings
import utils

needs = {'flags': ['apply', 'report', 'flags_of'],
         'run': ['generate_part', 'join_parts', 'read_batch', 'part_path',
                 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path'],
         'backends': ['USAGE', 'spent', 'record_usage', 'call_api'],
         'settings': ['MODELS', 'GENERATION', 'BATCHES_DIR']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)

## Model


In [ ]:
MODEL = 'mistral-small-2603'
PARTS = 5

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
have = len(utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR)))

print("Model:", MODEL, spec['provider'])
print("Pricing:", spec['price']['input'], spec['price']['output'])
print("Temperature:", settings.GENERATION['max_tokens'], settings.GENERATION['temperature'])
print("Collected:", have, wanted)
print("API Key:", bool(utils.api_key(spec['provider'])))

## Rerun


In [ ]:
FRESH = False

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print("Collected:", moved)
else:
    print("Resume Mode:")

## Cost


In [ ]:
OUTPUT_TOKENS = 300          # Output Tokens
INPUT_TOKENS = 24

left = wanted - have
price = spec['price']
cost = (left * INPUT_TOKENS * price['input']
        + left * OUTPUT_TOKENS * price['output']) / 1e6
print("Outstanding:", left, OUTPUT_TOKENS)
print("Cost:", cost, cost / PARTS)

## Generation


In [ ]:
# Part Runner
totals = {'read': 0, 'failed': 0, 'truncated': 0, 'repeated': 0,
          'blocked': 0, 'input': 0, 'output': 0, 'cost': 0.0}

def run_part(part):
    path, asked, failures = run.generate_part(MODEL, part, PARTS)
    if not asked:
        print("Complete:", part, PARTS)
        return
    # Empty Part
    if not path.exists():
        print("Part:", part, PARTS)
        return

    # Usage Accounting
    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed, truncated, repeated, blocked = run.read_batch(MODEL, path)
    usage, cost = dict(backends.USAGE), backends.spent(MODEL)
    for name, value in [('read', read), ('failed', failed),
                        ('truncated', truncated), ('repeated', repeated),
                        ('blocked', blocked), ('input', usage['input']),
                        ('output', usage['output']), ('cost', cost)]:
        totals[name] += value

    print("Part:", part, PARTS)
    print("Replies Read:", read, failed)
    print("Token Usage:", usage['input'], usage['output'])
    print("Cost:", cost)
    print("Mean Tokens:", usage['output'] / max(read - failed, 1))

print("Part:", PARTS, -(-wanted // PARTS))

In [ ]:
run_part(1)

In [ ]:
run_part(2)

In [ ]:
run_part(3)

In [ ]:
run_part(4)

In [ ]:
run_part(5)

## Merge


In [ ]:
joined, lines = run.join_parts(MODEL, PARTS)
print("Merged Rows:", lines, joined.name)

print("Merge:")
print("Replies Read:", totals['read'], totals['failed'])
print("Token Usage:", totals['input'], totals['output'])
print("Cost:", totals['cost'])
print("Mean Tokens:", totals['output'] / max(totals['read'] - totals['failed'], 1))

## Validation


In [ ]:
# Update Flags
flags.apply(MODEL)
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print("Collected:", MODEL)
else:
    marked = lambda name: collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['True', 'true']) \
        | ~collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['', 'False', 'false', 'nan'])
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print("Replies:", len(collected), blank)
    print("Coverage:", collected['prompt_id'].nunique(), len(prompts))

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))